In [ ]:
# run this notebook after 4_analyze_snipar_out_R
# use this notebook to subset the AoU VDS to a Hail MT
# after this, run 6_find_differences_twins_python

In [ ]:
import hail as hl
import os
import pandas
from google.cloud import storage
import re
import numpy as np 

In [ ]:
bucket=os.getenv('WORKSPACE_BUCKET')
hl.init(idempotent=True)
hl.default_reference('GRCh38')

In [ ]:
f_twins = bucket + "/relatedness/twin_dup.txt"
twins_all = pandas.read_table(f_twins)

#all twin samples
samples_list1 = twins_all.loc[:,'IID1'].tolist()
samples_list2 = twins_all.loc[:,'IID2'].tolist()
samples_list = samples_list1 + samples_list2
samples_list = list(set(samples_list))
samples_twins = [str(element) for element in samples_list]

f_sibs = bucket + "/relatedness/sibs_final.txt"
sibs_all = pandas.read_table(f_sibs)

#all sib samples
samples_list1 = sibs_all.loc[:,'idv1'].tolist()
samples_list2 = sibs_all.loc[:,'idv2'].tolist()
samples_list = samples_list1 + samples_list2
samples_list = list(set(samples_list))
samples_sibs = [str(element) for element in samples_list]


f_trios = bucket + "/relatedness/trios_final.txt"
trios_all = pandas.read_table(f_trios)
samples_list1 = trios_all.loc[:,'par1'].tolist()
samples_list2 = trios_all.loc[:,'par2'].tolist()
samples_list3 = trios_all.loc[:,'off'].tolist()
samples_list = samples_list1 + samples_list2 + samples_list3
samples_list = list(set(samples_list))
samples_trios = [str(element) for element in samples_list]

samples = samples_twins + samples_sibs + samples_trios
samples = list(set(samples))


In [ ]:
# read in vds 
vds_srwgs_path = os.getenv("WGS_VDS_PATH")
vds = hl.vds.read_vds(vds_srwgs_path)
vds = hl.vds.filter_samples(vds, samples, keep = True, remove_dead_alleles = True)

mt = vds.variant_data.annotate_entries(AD = hl.vds.local_to_global(vds.variant_data.LAD, 
                                                                   vds.variant_data.LA, 
                                                                   n_alleles=hl.len(vds.variant_data.alleles), 
                                                                   fill_value=0, number='R'))

mt = mt.annotate_entries(GT = hl.vds.lgt_to_gt(mt.LGT, mt.LA))

mt = mt.transmute_entries(FT = hl.if_else(mt.FT, "PASS", "FAIL"))
mt = hl.vds.to_dense_mt(hl.vds.VariantDataset(vds.reference_data, mt))

fields_to_drop_list = ['as_vets','as_vqsr','LAD', 'LGT', 'LA',
            'tranche_data', 'truth_sensitivity_snp_threshold', 
             'truth_sensitivity_indel_threshold','snp_vqslod_threshold','indel_vqslod_threshold']

mt = mt.drop(*(f for f in fields_to_drop_list if f in mt.entry or f in mt.row or f in mt.col or f in mt.globals))

out_path = f'{bucket}/data/twins_dups_sibs_trios_rep.mt'
mt = mt.filter_rows(hl.len(mt.filters) == 0) # vat filter passing 
mt_fin = mt.repartition(5000)
mt_fin.write(out_path, overwrite = True)

In [ ]:
# subset to twins
mt_path = f'{bucket}/data/twins_dups_sibs_trios_rep.mt'
out_path = f'{bucket}/data/twins_dups_rep_clean.mt'
mt = hl.read_matrix_table(mt_path)
mt_subset = mt.filter_cols(hl.literal(samples_twins).contains(mt.s))  
mt_subset.write(out_path, overwrite = True)

In [ ]:
#further subset 
mt_path = f'{bucket}/data/twins_dups_rep_clean.mt'
mt = hl.read_matrix_table(mt_path)
mt = mt.annotate_entries(GT = hl.or_missing(mt.GQ >= 20, mt.GT))
# split to 10 
split = 10 
size = len(twins_all) // split
i = 0 
num_split = 1

while i < len(twins_all):
    sub = []
    while len(sub) < size and i < len(twins_all):
        sub.append(i)
        i += 1
    print(sub)
    print(num_split)
    twins_sub = twins_all.iloc[sub]
    samples_list1 = twins_sub.loc[:,'IID1'].tolist()
    samples_list2 = twins_sub.loc[:,'IID2'].tolist()
    samples_list = samples_list1 + samples_list2
    samples_list = list(set(samples_list))
    samples_twins = [str(element) for element in samples_list]
    mt_subset = mt.filter_cols(hl.literal(samples_twins).contains(mt.s))  
    mt_subset = mt_subset.annotate_rows(is_non_ref = hl.agg.sum((mt_subset.GT.is_non_ref())))    
    mt_subset = mt_subset.filter_rows(mt_subset.is_non_ref == 0, keep = False)
    out_path = f'{bucket}/data/twins_dups_rep_samples_{num_split}.mt'
    mt_subset.write(out_path, overwrite = True)
    out_twins_sub = f'{bucket}/data/twins_dups_samples_{num_split}.txt'
    twins_sub.to_csv(out_twins_sub, index=False)
    num_split = num_split + 1

In [ ]:
# subset to sibs 
mt_path = f'{bucket}/data/twins_dups_sibs_trios_rep.mt'
out_path = f'{bucket}/data/sibs_rep_clean.mt'
mt = hl.read_matrix_table(mt_path)
mt_subset = mt.filter_cols(hl.literal(samples_sibs).contains(mt.s))
# remove things where there are no nonrefs
mt_subset = mt_subset.annotate_rows(is_non_ref = hl.agg.sum((mt_subset.GT.is_non_ref())))    
mt_subset = mt_subset.filter_rows(mt_subset.is_non_ref == 0, keep = False)

# remove things where there are no high quality calls
mt_subset = mt_subset.annotate_entries(GT = hl.or_missing(mt_subset.GQ >= 20, mt_subset.GT))
mt_subset = mt_subset.filter_rows(hl.agg.sum(hl.is_defined(mt_subset.GT)) == 0, keep = False)

# remove things where no variants pass filters 
mt_subset = mt_subset.filter_rows(hl.agg.sum(mt_subset.FT == "PASS") == 0, keep = False)
mt_subset.write(out_path, overwrite=True)

In [ ]:
# further subset 
mt_path = f'{bucket}/data/sibs_rep_clean.mt'
mt = hl.read_matrix_table(mt_path)

# split to 20 and clean 
split = 20 
size = len(sibs_all) // split
i = 0 
num_split = 1

while i < len(sibs_all):
    sub = []
    while len(sub) < size and i < len(sibs_all):
        sub.append(i)
        i += 1
    print(sub)
    print(num_split)
    sibs_sub = sibs_all.iloc[sub]
    samples_list1 = sibs_sub.loc[:,'idv1'].tolist()
    samples_list2 = sibs_sub.loc[:,'idv2'].tolist()
    samples_list = samples_list1 + samples_list2
    samples_list = list(set(samples_list))
    samples_sibs = [str(element) for element in samples_list]
    mt_subset = mt.filter_cols(hl.literal(samples_sibs).contains(mt.s))  
    
    # remove things where there are no nonrefs
    mt_subset = mt_subset.annotate_rows(is_non_ref = hl.agg.sum((mt_subset.GT.is_non_ref())))    
    mt_subset = mt_subset.filter_rows(mt_subset.is_non_ref == 0, keep = False)

    # remove things where there are no high quality calls
    mt_subset = mt_subset.annotate_entries(GT = hl.or_missing(mt_subset.GQ >= 20, mt_subset.GT))
    mt_subset = mt_subset.filter_rows(hl.agg.sum(hl.is_defined(mt_subset.GT)) == 0, keep = False)

    # remove things where no variants pass filters 
    mt_subset = mt_subset.filter_rows(hl.agg.sum(mt_subset.FT == "PASS") == 0, keep = False)
 
    out_path = f'{bucket}/data/sibs_samples_{num_split}.mt'
    mt_subset.write(out_path, overwrite = True)
    out_sibs_sub = f'{bucket}/data/sibs_samples_{num_split}.txt'
    sibs_sub.to_csv(out_sibs_sub, index=False)
    num_split = num_split + 1

In [ ]:
# subset to trios
mt_path = f'{bucket}/data/twins_dups_sibs_trios_rep.mt'
out_path = f'{bucket}/data/trios_rep_clean.mt'
mt = hl.read_matrix_table(mt_path)
mt_subset = mt.filter_cols(hl.literal(samples_trios).contains(mt.s))
# remove things where there are no nonrefs
mt_subset = mt_subset.annotate_rows(is_non_ref = hl.agg.sum((mt_subset.GT.is_non_ref())))    
mt_subset = mt_subset.filter_rows(mt_subset.is_non_ref == 0, keep = False)

# remove things where there are no high quality calls
mt_subset = mt_subset.annotate_entries(GT = hl.or_missing(mt_subset.GQ >= 20, mt_subset.GT))
mt_subset = mt_subset.filter_rows(hl.agg.sum(hl.is_defined(mt_subset.GT)) == 0, keep = False)

# remove things where no variants pass filters 
mt_subset = mt_subset.filter_rows(hl.agg.sum(mt_subset.FT == "PASS") == 0, keep = False)
mt_subset.write(out_path, overwrite=True)

In [ ]:
# further subset
mt_path = f'{bucket}/data/trios_rep_clean.mt'
mt = hl.read_matrix_table(mt_path)

trios_filtered = trios_all

# split to 20 and clean 
split = 10 
size = len(trios_filtered) // split
num_split = 1
i = size * (num_split - 1)

while i < len(trios_filtered):
    sub = []
    while len(sub) < size and i < len(trios_filtered):
        sub.append(i)
        i += 1
    print(sub)
    print(num_split)
    trios_sub = trios_filtered.iloc[sub]
    samples_list1 = trios_sub.loc[:,'par1'].tolist()
    samples_list2 = trios_sub.loc[:,'par2'].tolist()
    samples_list3 = trios_sub.loc[:,'off'].tolist()
    samples_list = samples_list1 + samples_list2 + samples_list3
    samples_list = list(set(samples_list))
    samples_trios = [str(element) for element in samples_list]
    mt_subset = mt.filter_cols(hl.literal(samples_trios).contains(mt.s))  
    
    # remove things where there are no nonrefs
    mt_subset = mt_subset.annotate_rows(is_non_ref = hl.agg.sum((mt_subset.GT.is_non_ref())))    
    mt_subset = mt_subset.filter_rows(mt_subset.is_non_ref == 0, keep = False)

    # remove things where there are no high quality calls
    mt_subset = mt_subset.annotate_entries(GT = hl.or_missing(mt_subset.GQ >= 20, mt_subset.GT))
    mt_subset = mt_subset.filter_rows(hl.agg.sum(hl.is_defined(mt_subset.GT)) == 0, keep = False)

    # remove things where no variants pass filters 
    mt_subset = mt_subset.filter_rows(hl.agg.sum(mt_subset.FT == "PASS") == 0, keep = False)
 
    out_path = f'{bucket}/data/trios_samples_{num_split}.mt'
    mt_subset.write(out_path, overwrite = True)
    out_trios_sub = f'{bucket}/data/trios_samples_{num_split}.txt'
    trios_sub.to_csv(out_trios_sub, index=False)
    num_split = num_split + 1